In [3]:
import numpy as np
import matplotlib.pyplot as plt

# Generate 3D dataset for Chapter 8
np.random.seed(4)
m = 60
w1, w2 = 0.1, 0.3
noise = 0.1

angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m)
X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m)
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)

print("X shape:", X.shape)
print("Ready")

X shape: (60, 3)
Ready


In [4]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 2)
X2D = pca.fit_transform(X)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (48, 3)
X_test shape: (12, 3)


In [8]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1
print("Components needed for 95% variance:", d)

Components needed for 95% variance: 2


In [9]:
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

In [11]:
from sklearn.decomposition import PCA

# Use correct n_components for our 3D data
# Can't have more components than features!
pca = PCA(n_components=2)  # 3D → 2D
X_reduced = pca.fit_transform(X_train)
X_recovered = pca.inverse_transform(X_reduced)

print("Original shape:", X_train.shape)
print("Reduced shape:", X_reduced.shape)
print("Recovered shape:", X_recovered.shape)
print("Variance retained:", pca.explained_variance_ratio_.sum())

Original shape: (48, 3)
Reduced shape: (48, 2)
Recovered shape: (48, 3)
Variance retained: 0.9887499625254802


In [13]:
rnd_pca = PCA(n_components=2, svd_solver="randomized")
X_reduced = rnd_pca.fit_transform(X_train)

In [15]:
from sklearn.decomposition import IncrementalPCA
n_batches = 2
inc_pca = IncrementalPCA(n_components=2)
for X_batch in np.array_split(X_train, n_batches):
 inc_pca.partial_fit(X_batch)
X_reduced = inc_pca.transform(X_train)

In [17]:
# This cell demonstrates memory-mapped PCA for large datasets
# We'll simulate it with our existing data instead

m, n = X_train.shape  # get dimensions

filename = "my_mnist.data"  # dummy filename

# Save and reload as memmap (simulating large dataset on disk)
X_mm = np.memmap(filename, dtype="float32", mode="write", shape=(m, n))
X_mm[:] = X_train
del X_mm  # flush to disk

# Now read it back
X_mm = np.memmap(filename, dtype="float32", mode="readonly", shape=(m, n))
n_batches = 2
batch_size = m // n_batches
inc_pca = IncrementalPCA(n_components=2, batch_size=batch_size)
inc_pca.fit(X_mm)

print("Incremental PCA on memory-mapped data done")
print("Variance retained:", inc_pca.explained_variance_ratio_.sum())

Incremental PCA on memory-mapped data done
Variance retained: 0.9887456153126215


In [18]:
from sklearn.decomposition import KernelPCA
rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.04)
X_reduced = rbf_pca.fit_transform(X)

In [21]:
from sklearn.datasets import make_swiss_roll

# Generate Swiss roll dataset with labels for classification
X_swiss, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
y = (t > 6.9).astype(int)  # binary labels

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_swiss, y, random_state=42)

print("X shape:", X_swiss.shape)
print("y shape:", y.shape)

X shape: (1000, 3)
y shape: (1000,)


In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.decomposition import KernelPCA

clf = Pipeline([
    ("kpca", KernelPCA(n_components=2)),
    ("log_reg", LogisticRegression())
])

param_grid = [{
    "kpca__gamma": np.linspace(0.03, 0.05, 10),
    "kpca__kernel": ["rbf", "sigmoid"]
}]

grid_search = GridSearchCV(clf, param_grid, cv=3)
grid_search.fit(X_swiss, y)
print("Best params:", grid_search.best_params_)

Best params: {'kpca__gamma': 0.043333333333333335, 'kpca__kernel': 'rbf'}


In [23]:
rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.0433,
 fit_inverse_transform=True)
X_reduced = rbf_pca.fit_transform(X)
X_preimage = rbf_pca.inverse_transform(X_reduced)

In [24]:
from sklearn.metrics import mean_squared_error

In [25]:
mean_squared_error(X, X_preimage)

0.19302738468182745

In [26]:
from sklearn.manifold import LocallyLinearEmbedding
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_reduced = lle.fit_transform(X)